# LP Feed Optimizer

### Least cost dairy ration formulation for Gujarat and Western Maharashtra

Linear programming based feed optimization for dairy cattle using nutritional constraints and ingredient availability.

## Objective

Formulate a nutritionally balanced ration at minimum cost while satisfying the daily requirements for:

| Nutrient | Role |
|---|---|
| DMI | Intake control |
| CP | Protein supply |
| ME | Energy supply |
| Ca | Mineral balance |
| P | Mineral balance |
| NDF | Fibre balance |

The model supports both predefined breed profiles and custom animal inputs.

## Input Files

| File | Use |
|---|---|
| `nutri_matrix_guj.csv` | Ingredient nutrient matrix |
| `animal_profiles.csv` | Breed profiles and targets |

## Animal Profiles

| Breed | BW kg | Milk kg/day | Fat % | Notes |
|---|---:|---:|---:|---|
| HF_crossbreed | 450 | 10.0 | 4.0 | High yield crossbred |
| Gir | 390 | 7.0 | 4.6 | Indigenous dairy breed |
| Jersey_cross | 350 | 9.0 | 5.0 | Higher fat, moderate yield |
| Kankrej | 420 | 5.5 | 4.2 | Dual purpose indigenous breed |
| Sahiwal | 430 | 10.0 | 4.8 | Strong indigenous dairy breed |

Custom profiles can also be generated from body weight, milk yield, and milk fat.

## Model Assumptions

| Assumption | Value |
|---|---|
| NPN CP fraction | ≤ 30% |
| Concentrate fraction | ≤ 60% of DMI |
| Forage NDF | ≥ 75% of total NDF |
| Ca:P ratio | 1.5 to 2.0 |
| Crossbred NDF range | 28% to 40% |
| Indigenous NDF range | 30% to 42% |


## Imports and setup

In [1]:
import argparse
import csv
import sys
from pathlib import Path
import pulp
import pandas as pd

In [2]:
_DIR = Path.cwd()
DEFAULT_MATRIX  = _DIR / "nutri_matrix_guj.csv"
DEFAULT_ANIMALS = _DIR / "animal_profiles.csv"
MCAL_TO_MJ          = 4.184   # 1 Mcal = 4.184 MJ (thermochemical calorie)
NPN_CP_FRAC_MAX     = 0.30    # ICAR: urea-derived CP <= 30% of total CP
CONC_DMI_FRAC_MAX   = 0.60    # ICAR 2013: concentrates <= 1/3 of DMI (strict = 33%;
                               # using 60% as soft upper since roughage floor already enforces forage NDF)

In [3]:
ICAR_MAINT = [
    (400, 8.64, 11.82, 436, 18, 8),
    (450, 9.72, 12.94, 476, 20, 9),
    (500, 10.80, 14.04, 515, 23, 10),
    (550, 11.88, 15.10, 553, 25, 11),
    (600, 12.96, 16.15, 591, 27, 12),
]

# ICAR 2013 production per kg milk — (fat_pct, DM_kg, ME_Mcal, CP_g, Ca_g, P_g)
ICAR_PROD = [
    (4.0, 0.510, 1.20, 96, 3.2, 1.8),
    (6.0, 0.670, 1.58, 124, 4.8, 1.8),
]

## Requirement Calculation

The requirement functions interpolate maintenance and production needs, then compute daily nutrient targets and intake bounds.

In [4]:
def _interp_maint(bw_kg):
    """Linearly interpolate ICAR 2013 maintenance table for given BW."""
    if bw_kg <= ICAR_MAINT[0][0]:
        # Below range: scale proportionally
        b0, d0, m0, c0, ca0, p0 = ICAR_MAINT[0]
        f = bw_kg / b0
        return d0*f, m0*f, c0*f, ca0*f, p0*f
    if bw_kg >= ICAR_MAINT[-1][0]:
        b1, d1, m1, c1, ca1, p1 = ICAR_MAINT[-2]
        b2, d2, m2, c2, ca2, p2 = ICAR_MAINT[-1]
        t = (bw_kg - b1) / (b2 - b1)
        return d1+(d2-d1)*t, m1+(m2-m1)*t, c1+(c2-c1)*t, ca1+(ca2-ca1)*t, p1+(p2-p1)*t
    for i in range(len(ICAR_MAINT) - 1):
        b1, d1, m1, c1, ca1, p1 = ICAR_MAINT[i]
        b2, d2, m2, c2, ca2, p2 = ICAR_MAINT[i+1]
        if b1 <= bw_kg <= b2:
            t = (bw_kg - b1) / (b2 - b1)
            return d1+(d2-d1)*t, m1+(m2-m1)*t, c1+(c2-c1)*t, ca1+(ca2-ca1)*t, p1+(p2-p1)*t


def _interp_prod(fat_pct):
    """Linearly interpolate ICAR 2013 production requirement per kg milk for given fat%."""
    fat_pct = max(ICAR_PROD[0][0], min(ICAR_PROD[-1][0], fat_pct))
    f1, d1, m1, c1, ca1, p1 = ICAR_PROD[0]
    f2, d2, m2, c2, ca2, p2 = ICAR_PROD[-1]
    t = (fat_pct - f1) / (f2 - f1)
    return d1+(d2-d1)*t, m1+(m2-m1)*t, c1+(c2-c1)*t, ca1+(ca2-ca1)*t, p1  # P fixed at 1.8


def compute_requirements(bw_kg, milk_kg_day, fat_pct, dmi_type="crossbred"):
    """
    Compute daily nutrient requirements using ICAR 2013 factorial method.
    dmi_type: 'crossbred' => DMI 2.5-3.0% BW; 'indigenous' => 2.0-2.5% BW
    All values confirmed against ICAR 2013 worked example:
      450kg cow, 10kg/day, 4% fat -> CP=1436g, Ca=52g, P=27g, ME=24.94 Mcal=104.35 MJ
    """
    dm_m, me_m, cp_m, ca_m, p_m = _interp_maint(bw_kg)
    dm_l, me_l, cp_l, ca_l, p_l = _interp_prod(fat_pct)

    dm_total = dm_m + dm_l * milk_kg_day
    me_total = (me_m + me_l * milk_kg_day) * MCAL_TO_MJ  # Convert Mcal to MJ
    cp_total = cp_m + cp_l * milk_kg_day
    ca_total = ca_m + ca_l * milk_kg_day
    p_total  = p_m  + p_l  * milk_kg_day

    # DMI target range
    # Allow a practical tolerance band
    dmi_min = dm_total * 0.92
    dmi_max = dm_total * 1.10

    return {
        "breed": f"custom_{bw_kg}kg_{milk_kg_day}L_{fat_pct}fat",
        "description": f"Custom: {bw_kg} kg BW, {milk_kg_day} kg/day milk, {fat_pct}% fat ({dmi_type})",
        "bw_kg": bw_kg,
        "milk_yield_kg_day": milk_kg_day,
        "milk_fat_pct": fat_pct,
        "dmi_min_kg": round(dmi_min, 2),
        "dmi_max_kg": round(dmi_max, 2),
        "cp_req_g_day": round(cp_total, 1),
        "me_req_mj_day": round(me_total, 1),
        "ca_req_g_day": round(ca_total, 1),
        "p_req_g_day": round(p_total, 1),
        # NDF range by breed type
        "ndf_min_frac": 0.30 if dmi_type == "indigenous" else 0.28,
        "ndf_max_frac": 0.42 if dmi_type == "indigenous" else 0.40,
        "forage_ndf_min_frac": 0.75,
        "ca_p_ratio_min": 1.5,
        "ca_p_ratio_max": 2.0,
        "dmi_type": dmi_type,
    }

## Data Loading

In [5]:
def load_matrix(path):
    rows = []
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            clean = {}
            for k, v in row.items():
                k = k.strip(); v = v.strip()
                try:
                    clean[k] = float(v)
                except ValueError:
                    clean[k] = v
            rows.append(clean)
    return rows


def load_animals(path):
    profiles = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            breed = row["breed"].strip()
            p = {}
            for k, v in row.items():
                k = k.strip(); v = v.strip()
                try:
                    p[k] = float(v)
                except ValueError:
                    p[k] = v
            profiles[breed] = p
    return profiles

## Linear Programming Formulation

| Constraint | Purpose |
|---|---|
| DMI | Intake bounds |
| CP | Protein requirement |
| ME | Energy requirement |
| Ca | Calcium requirement |
| P | Phosphorus requirement |
| NDF | Fibre bounds |
| Forage NDF | Rumen health |
| Ca:P | Mineral balance |
| NPN fraction | Urea safety |
| Concentrate fraction | Rumen safety |

The solver minimizes daily feed cost subject to these constraints.

In [6]:
def solve(animal, ingredients):
    """
    Least-cost LP ration formulation.

    Decision variable: x_i = kg as-fed per day of ingredient i

    Objective: minimize  sum_i( cost_inr_per_kg_asfed_i * x_i )

    TRIVIAL CONSTRAINTS
      T1  x_i >= inclusion_min_kg_i          (non-negativity + mandatory minimums)
      T2  x_i <= inclusion_max_kg_i          (safety / palatability caps)

    ANIMAL CONSTRAINTS (ICAR 2013)
      A1  sum_i(x_i * DM_i)                  >= dmi_min_kg
      A2  sum_i(x_i * DM_i)                  <= dmi_max_kg
      A3  sum_i(x_i * DM_i * CP_i/100)*1000  >= cp_req_g_day
      A4  sum_i(x_i * DM_i * ME_i)           >= me_req_mj_day
      A5  sum_i(x_i * DM_i * Ca_i/100)*1000  >= ca_req_g_day
      A6  sum_i(x_i * DM_i * P_i/100)*1000   >= p_req_g_day
      A7  NDF_total >= ndf_min_frac * DMI_total
      A8  NDF_total <= ndf_max_frac * DMI_total
      A9  NDF_roughage >= forage_ndf_min_frac * NDF_total
      A10 Ca_total >= ca_p_ratio_min * P_total
      A11 Ca_total <= ca_p_ratio_max * P_total

    INGREDIENT-SPECIFIC CONSTRAINTS
      I1  NPN_CP_total <= NPN_CP_FRAC_MAX * CP_total   (urea/NPN rumen safety, ICAR)
      I2  Conc_DM_total <= CONC_DMI_FRAC_MAX * DMI_total  (rumen health, ICAR 2013)
    """
    prob = pulp.LpProblem("low_cost_feed", pulp.LpMinimize)

    x = {
        ing["name"]: pulp.LpVariable(
            f"x_{ing['ingredient_id']:.0f}",
            lowBound=float(ing["inclusion_min_kg"]),
            upBound=float(ing["inclusion_max_kg"]),
        )
        for ing in ingredients
    }

    def dm_expr(ing):
        return ing["dm_pct"] / 100.0 * x[ing["name"]]

    prob += pulp.lpSum(ing["cost_inr_per_kg_asfed"] * x[ing["name"]] for ing in ingredients), "Cost"

    DMI = pulp.lpSum(dm_expr(ing) for ing in ingredients)
    prob += DMI >= animal["dmi_min_kg"], "DMI_min"
    prob += DMI <= animal["dmi_max_kg"], "DMI_max"
    CP = pulp.lpSum(dm_expr(ing) * ing["cp_pct_dm"] / 100 * 1000 for ing in ingredients)
    prob += CP >= animal["cp_req_g_day"], "CP_min"
    ME = pulp.lpSum(dm_expr(ing) * ing["me_mj_per_kg_dm"] for ing in ingredients)
    prob += ME >= animal["me_req_mj_day"], "ME_min"
    Ca = pulp.lpSum(dm_expr(ing) * ing["ca_pct_dm"] / 100 * 1000 for ing in ingredients)
    prob += Ca >= animal["ca_req_g_day"], "Ca_min"
    P = pulp.lpSum(dm_expr(ing) * ing["p_pct_dm"] / 100 * 1000 for ing in ingredients)
    prob += P >= animal["p_req_g_day"], "P_min"
    roughage_cats = {"roughage_dry", "roughage_green"}
    NDF       = pulp.lpSum(dm_expr(ing) * ing["ndf_pct_dm"] / 100 for ing in ingredients)
    NDF_rough = pulp.lpSum(dm_expr(ing) * ing["ndf_pct_dm"] / 100
                           for ing in ingredients if ing["category"] in roughage_cats)
    prob += NDF >= animal["ndf_min_frac"] * DMI, "NDF_min"
    prob += NDF <= animal["ndf_max_frac"] * DMI, "NDF_max"
    prob += NDF_rough >= animal["forage_ndf_min_frac"] * NDF, "ForageNDF_min"
    prob += Ca >= animal["ca_p_ratio_min"] * P, "CaP_min"
    prob += Ca <= animal["ca_p_ratio_max"] * P, "CaP_max"
    NPN_CP = pulp.lpSum(dm_expr(ing) * ing["cp_pct_dm"] / 100 * 1000
                        for ing in ingredients if ing["category"] == "NPN")
    prob += NPN_CP <= NPN_CP_FRAC_MAX * CP, "NPN_frac"
    conc_cats = {"concentrate_energy", "concentrate_protein"}
    Conc_DM = pulp.lpSum(dm_expr(ing) for ing in ingredients if ing["category"] in conc_cats)
    prob += Conc_DM <= CONC_DMI_FRAC_MAX * DMI, "Conc_frac"

    status = prob.solve(pulp.PULP_CBC_CMD(msg=0))
    return prob, status, x

## Result Reporting

In [7]:
def print_results(prob, x, animal, ingredients):
    print("\n" + "=" * 70)
    print("  Feed Optimization Results")
    print("=" * 70)
    print(f"  Animal   : {animal.get('description', animal['breed'])}")
    print(f"  BW       : {animal['bw_kg']:.0f} kg | Milk: {animal['milk_yield_kg_day']} kg/day"
          f" | Fat: {animal['milk_fat_pct']}%")
    print(f"  Status   : {pulp.LpStatus[prob.status]}")

    if pulp.LpStatus[prob.status] != "Optimal":
        print("\n  Infeasible — check ingredient availability or relax constraints.\n")
        return

    sol = [(ing, pulp.value(x[ing["name"]])) for ing in ingredients
           if pulp.value(x[ing["name"]]) is not None and pulp.value(x[ing["name"]]) > 1e-4]

    cost_total = pulp.value(prob.objective)

    # Actuals
    dmi   = sum(ing["dm_pct"]/100 * q for ing, q in sol)
    cp    = sum(ing["dm_pct"]/100 * ing["cp_pct_dm"]/100 * q * 1000 for ing, q in sol)
    me    = sum(ing["dm_pct"]/100 * ing["me_mj_per_kg_dm"] * q for ing, q in sol)
    ca    = sum(ing["dm_pct"]/100 * ing["ca_pct_dm"]/100 * q * 1000 for ing, q in sol)
    p     = sum(ing["dm_pct"]/100 * ing["p_pct_dm"]/100 * q * 1000 for ing, q in sol)
    ndf   = sum(ing["dm_pct"]/100 * ing["ndf_pct_dm"]/100 * q for ing, q in sol)

    roughage_cats = {"roughage_dry", "roughage_green"}
    ndf_r = sum(ing["dm_pct"]/100 * ing["ndf_pct_dm"]/100 * q
                for ing, q in sol if ing["category"] in roughage_cats)

    print(f"\n{'Ingredient':<32} {'As-Fed':>8} {'DM':>7} {'Cost':>9} {'Category'}")
    print("-" * 75)
    for ing, q in sorted(sol, key=lambda t: t[0]["category"]):
        cost_i = ing["cost_inr_per_kg_asfed"] * q
        dm_i   = ing["dm_pct"] / 100 * q
        print(f"  {ing['name']:<30} {q:>8.3f} {dm_i:>7.3f} {cost_i:>9.2f}  {ing['category']}")
    print("-" * 75)
    print(f"  {'TOTAL':<30} {'':>8} {dmi:>7.3f} {cost_total:>9.2f}")

    EPS = 0.005  # 0.5% tolerance for float comparison

    def chk(achieved, req, ge=True):
        return "OK" if (achieved >= req * (1 - EPS) if ge else achieved <= req * (1 + EPS)) else "FAIL"

    print(f"\n{'Nutrient / Constraint':<32} {'Achieved':>12} {'Required':>12}  Status  Source")
    print("-" * 80)

    # ICAR 2013 requirements
    src = "ICAR 2013"
    print(f"  {'DMI min (kg/day)':<30} {dmi:>12.3f} {animal['dmi_min_kg']:>12.2f}  "
          f"{chk(dmi, animal['dmi_min_kg']):>6}  {src}")
    print(f"  {'DMI max (kg/day)':<30} {animal['dmi_max_kg']:>12.2f} {dmi:>12.3f}  "
          f"{chk(animal['dmi_max_kg'], dmi):>6}  {src}")
    print(f"  {'CP (g/day)':<30} {cp:>12.1f} {animal['cp_req_g_day']:>12.1f}  "
          f"{chk(cp, animal['cp_req_g_day']):>6}  {src}")
    print(f"  {'ME (MJ/day)':<30} {me:>12.2f} {animal['me_req_mj_day']:>12.1f}  "
          f"{chk(me, animal['me_req_mj_day']):>6}  {src}")
    print(f"  {'Ca (g/day)':<30} {ca:>12.2f} {animal['ca_req_g_day']:>12.1f}  "
          f"{chk(ca, animal['ca_req_g_day']):>6}  {src}")
    print(f"  {'P (g/day)':<30} {p:>12.2f} {animal['p_req_g_day']:>12.1f}  "
          f"{chk(p, animal['p_req_g_day']):>6}  {src}")

    ndf_frac = ndf / dmi if dmi > 0 else 0
    ndf_ok = animal["ndf_min_frac"] * (1-EPS) <= ndf_frac <= animal["ndf_max_frac"] * (1+EPS)
    print(f"  {'NDF % of DMI':<30} {ndf_frac*100:>11.1f}% "
          f"[{animal['ndf_min_frac']*100:.0f}-{animal['ndf_max_frac']*100:.0f}%]  "
          f"{'OK' if ndf_ok else 'FAIL':>6}  ICAR/NRC")

    fndf_frac = ndf_r / ndf if ndf > 0 else 0
    print(f"  {'Forage NDF % of NDF':<30} {fndf_frac*100:>11.1f}% "
          f"[>={animal['forage_ndf_min_frac']*100:.0f}%]   "
          f"{chk(fndf_frac, animal['forage_ndf_min_frac']):>6}  ICAR 2013")

    ca_p = ca / p if p > 0 else 0
    cap_ok = (animal["ca_p_ratio_min"] * (1-EPS)) <= ca_p <= (animal["ca_p_ratio_max"] * (1+EPS))
    print(f"  {'Ca:P ratio':<30} {ca_p:>12.2f} "
          f"[{animal['ca_p_ratio_min']:.1f}-{animal['ca_p_ratio_max']:.1f}]   "
          f"{'OK' if cap_ok else 'FAIL':>6}  ICAR 2013")

    print(f"\n  Total daily cost : INR {cost_total:.2f}")
    print(f"  Cost per kg milk : INR {cost_total / animal['milk_yield_kg_day']:.2f}")
    print("=" * 70 + "\n")

## CLI Helpers

The command line entry points are retained for terminal use, but the notebook examples below are the intended workflow.

In [8]:
def list_breeds(profiles):
    print("\nAvailable animal profiles (source: ICAR 2013 factorial method):")
    print(f"  {'Breed':<22} {'BW':>6} {'Milk':>7} {'Fat':>5}  {'DMI min-max':^13}  {'CP':>6}  {'ME':>7}")
    print("-" * 85)
    for b, p in profiles.items():
        print(f"  {b:<22} {p['bw_kg']:>6.0f} {p['milk_yield_kg_day']:>7.1f} {p['milk_fat_pct']:>5.1f}"
              f"  {p['dmi_min_kg']:.1f}-{p['dmi_max_kg']:.1f} kg   "
              f"{p['cp_req_g_day']:>6.0f}g  {p['me_req_mj_day']:>6.1f} MJ")
    print()


## Input Preview

Use this section to inspect the working files before solving a ration.

In [9]:
df = pd.read_csv(DEFAULT_MATRIX)
df.set_index('ingredient_id', inplace=True)
df.info()

df2 = pd.read_csv(DEFAULT_ANIMALS)
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 1 to 19
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   name                   19 non-null     str    
 1   category               19 non-null     str    
 2   dm_pct                 19 non-null     float64
 3   cp_pct_dm              19 non-null     float64
 4   me_mj_per_kg_dm        19 non-null     float64
 5   ndf_pct_dm             19 non-null     float64
 6   adf_pct_dm             19 non-null     float64
 7   hemicellulose_pct_dm   19 non-null     float64
 8   cellulose_pct_dm       19 non-null     float64
 9   lignin_pct_dm          19 non-null     float64
 10  ee_pct_dm              19 non-null     float64
 11  ash_pct_dm             19 non-null     float64
 12  ca_pct_dm              19 non-null     float64
 13  p_pct_dm               19 non-null     float64
 14  tdn_pct_dm             19 non-null     float64
 15  cost_inr_per_kg_asf

In [10]:
df.head(19)

,name,category,dm_pct,cp_pct_dm,me_mj_per_kg_dm,ndf_pct_dm,adf_pct_dm,hemicellulose_pct_dm,cellulose_pct_dm,lignin_pct_dm,ee_pct_dm,ash_pct_dm,ca_pct_dm,p_pct_dm,tdn_pct_dm,cost_inr_per_kg_asfed,inclusion_min_kg,inclusion_max_kg
ingredient_id,,,,,,,,,,,,,,,,,,
1,Maize grain,concentrate_energy,92.71,9.51,11.51,26.15,6.64,19.51,4.91,1.50,4.43,3.01,0.000,0.300,89.00,22.0,0.00,4.00
2,Wheat bran,concentrate_energy,91.33,14.46,9.78,41.16,12.57,28.59,9.15,2.92,3.14,5.76,0.130,1.040,71.90,24.0,0.00,3.00
3,De-oiled rice bran (DORB),concentrate_energy,92.33,13.25,7.39,47.16,16.98,30.18,10.15,5.42,1.81,9.37,2.040,1.610,83.40,18.0,0.00,3.00
4,Molasses (sugarcane),concentrate_energy,76.00,5.00,12.21,0.00,0.00,0.00,0.00,0.00,0.00,10.00,1.100,0.080,75.00,14.0,0.00,1.00
5,Groundnut cake (expeller),concentrate_protein,93.18,41.54,9.44,24.17,18.47,5.70,13.59,3.59,6.88,5.78,0.200,0.650,77.00,44.0,0.00,2.50
6,Cottonseed cake (expeller),concentrate_protein,91.30,27.11,7.71,47.31,35.56,11.75,24.40,7.26,7.51,4.44,0.160,0.750,60.40,41.0,0.00,2.00
7,Mustard cake (expeller),concentrate_protein,92.77,37.29,9.21,23.33,17.73,5.60,13.77,3.23,7.03,5.11,0.680,0.990,74.30,42.0,0.00,1.50
8,Soybean meal (solvent),concentrate_protein,91.41,45.59,9.24,18.23,10.47,7.76,8.40,1.85,2.01,6.63,0.300,0.650,84.00,48.0,0.00,2.50
9,Wheat straw,roughage_dry,90.13,3.39,4.65,76.22,50.99,25.23,36.07,9.11,1.04,9.79,0.180,0.050,41.00,5.0,0.00,8.00


In [11]:
df2.head()

,breed,description,bw_kg,milk_yield_kg_day,milk_fat_pct,dmi_min_kg,dmi_max_kg,cp_req_g_day,me_req_mj_day,ca_req_g_day,p_req_g_day,ndf_min_frac,ndf_max_frac,forage_ndf_min_frac,ca_p_ratio_min,ca_p_ratio_max,dmi_type
0,HF_crossbreed,HF x Gir/Kankrej 50% exotic crossbreed — most ...,450,10.0,4.0,13.63,16.30,1436,104.3,52.0,27.0,0.28,0.40,0.75,1.5,2.0,crossbred
1,Gir,Pure Gir (Bos indicus) — indigenous milch bree...,390,7.0,4.6,11.34,13.56,1156,86.7,43.3,20.4,0.30,0.42,0.75,1.5,2.0,indigenous
2,Jersey_crossbreed,Jersey x Sahiwal/Gir 50% exotic crossbreed — c...,350,9.0,5.0,11.84,14.16,1372,95.6,51.8,23.2,0.28,0.40,0.75,1.5,2.0,crossbred
3,Kankrej,Pure Kankrej (dual-purpose Bos indicus) — nati...,420,5.5,4.2,11.01,13.16,995,79.8,37.3,18.3,0.30,0.42,0.75,1.5,2.0,indigenous
4,Sahiwal,Pure Sahiwal — high-yielding Bos indicus; occa...,430,10.0,4.8,13.83,16.53,1532,108.8,57.6,26.6,0.28,0.40,0.75,1.5,2.0,indigenous


## Example Usage

In [12]:
ingredients = load_matrix(DEFAULT_MATRIX)
profiles = load_animals(DEFAULT_ANIMALS)

animal = profiles["HF_crossbreed"]

prob, status, x = solve(animal, ingredients)
print_results(prob, x, animal, ingredients)


  Feed Optimization Results
  Animal   : HF x Gir/Kankrej 50% exotic crossbreed â€” most prevalent commercial dairy animal in Gujarat/Maharashtra cooperatives. BW and requirements per ICAR 2013 worked example (450kg 10kg milk 4% fat).
  BW       : 450 kg | Milk: 10.0 kg/day | Fat: 4.0%
  Status   : Optimal

Ingredient                         As-Fed      DM      Cost Category
---------------------------------------------------------------------------
  Urea                              0.150   0.148      6.97  NPN
  Maize grain                       4.000   3.708     88.00  concentrate_energy
  De-oiled rice bran (DORB)         0.059   0.054      1.06  concentrate_energy
  Molasses (sugarcane)              1.000   0.760     14.00  concentrate_energy
  Mustard cake (expeller)           1.500   1.392     63.00  concentrate_protein
  Soybean meal (solvent)            0.258   0.236     12.39  concentrate_protein
  Mineral-vitamin premix            0.116   0.114     13.32  mineral_supplemen